In [ ]:
# Extract ALFIN city/year observational units that are relevant for our work

import os
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/alfin_raw.parquet")
GEO_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/alfin_raw_geo.parquet")

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data/alfin_units.parquet")


In [96]:
# Load the data

df = pd.read_parquet(INPUT_FILEPATH)
geo_df = pd.read_parquet(GEO_FILEPATH)

# Merge on state names / abbrevs

state_fips_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "state_fips.csv"))
state_fips_df['STATE_FIPS'] = state_fips_df['STATE_FIPS'].astype(str).str.zfill(2)
geo_df = geo_df.merge(state_fips_df[['STATE_FIPS', 'STATE']], on='STATE_FIPS', how='left')

In [97]:
# Keep only cities, townships, and special districts

df = df.loc[df['UNIT_TYPE'].isin(['2', '3'])]

In [98]:
# Get list of city/year observational units

dfg = df[['ID', 'UNIT_TYPE', 'YEAR']].drop_duplicates()
dfg = dfg.merge(geo_df[['ID', 'YEAR', 'NAME', 'STATE', 'STATE_FIPS', 'COUNTY_FIPS']], on=['ID', 'YEAR'], how='left')
dfg = dfg.sort_values(by=['STATE_FIPS', 'COUNTY_FIPS', 'NAME', 'YEAR']).reset_index(drop=True)

# assert no duplicates

assert dfg[['NAME', 'STATE', 'COUNTY_FIPS', 'YEAR']].duplicated().sum() == 0

In [ ]:
dfg.to_parquet(OUTPUT_FILEPATH)

,ID,UNIT_TYPE,YEAR,NAME,STATE,STATE_FIPS,COUNTY_FIPS
0,01200100100000,2,2012,AUTAUGAVILLE TOWN,AL,01,001
1,012001122498,2,2017,AUTAUGAVILLE TOWN,AL,01,001
2,012001122498,2,2022,AUTAUGAVILLE TOWN,AL,01,001
3,01200100200000,2,2012,BILLINGSLEY TOWN,AL,01,001
4,012001160545,2,2017,BILLINGSLEY TOWN,AL,01,001
...,...,...,...,...,...,...,...
148021,51202300200000,2,2015,UPTON TOWN,WY,56,045
148022,51202300200000,2,2016,UPTON TOWN,WY,56,045
148023,562045115507,2,2017,UPTON TOWN,WY,56,045
148024,562045115507,2,2018,UPTON TOWN,WY,56,045
